# 🌿 Crop Disease Diagnostician — Full Pipeline

**Ek hi notebook mein sab kuch:** Data download → Training → Convert → Evaluate → Download

---

## ✅ Shuru karne se pehle (ZARURI):
1. **Runtime → Change runtime type → T4 GPU → Save**
2. Kaggle account se `kaggle.json` download karo:
   - [kaggle.com](https://kaggle.com) → Profile → Settings → API → **Create New Token**
3. **Sirf Step 1 run karo pehle** — runtime restart hogi, phir Step 2 se aage jaao

---
**Total time: ~75-80 minutes**
| Step | Time |
|---|---|
| Data download + organize | ~15 min |
| Model training | ~45 min |
| Convert + Evaluate | ~15 min |

---
# 📦 STEP 1: Install Libraries

⚠️ **Yeh cell runtime restart karega — bilkul normal hai!**

Restart ke baad **Step 2 se continue karo** — Step 1 dubara mat chalana.

In [ ]:
# Install all required packages
!pip install -q kaggle seaborn scikit-learn
!pip install -q tensorflowjs==4.15.0

# FIX: tensorflowjs==4.15.0 downgrades ml_dtypes to 0.3.2
# which breaks JAX and TensorFlow imports.
# We upgrade it back to a compatible version.
!pip install -q --upgrade ml_dtypes

print('✅ Libraries installed!')
print('🔄 Runtime restart ho raha hai...')
print('   Restart ke baad Step 2 se start karo. Step 1 dubara run mat karna!')

import time
time.sleep(2)

# Restart runtime so all packages load cleanly
import os
os.kill(os.getpid(), 9)

---
# 🔑 STEP 2: Imports + Upload kaggle.json

▶ **Yahaan se continue karo restart ke baad**

Neeche wala cell run karo → file picker aayega → apna `kaggle.json` upload karo

In [ ]:
import os, json, shutil, random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from google.colab import files

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

# Mixed precision for faster T4 training
tf.keras.mixed_precision.set_global_policy('mixed_float16')
print('Mixed precision: ON')

In [ ]:
print('👆 File picker aayega — apna kaggle.json upload karo:')
uploaded = files.upload()

os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('✅ Kaggle credentials set!')

---
# 🌱 STEP 3: Download PlantVillage Dataset from Kaggle
~1 GB download, takes ~5 minutes

In [ ]:
os.makedirs('/content/kaggle_download', exist_ok=True)

print('Downloading PlantVillage from Kaggle... (~5 min)')
!kaggle datasets download -d abdallahalidev/plantvillage-dataset \
    -p /content/kaggle_download --unzip

# Auto-detect the color images folder
COLOR_DIR = None
for root, dirs, _ in os.walk('/content/kaggle_download'):
    if any('Tomato' in d for d in dirs) and any('Corn' in d for d in dirs):
        COLOR_DIR = root
        break

# Fallback paths
if COLOR_DIR is None:
    for candidate in [
        '/content/kaggle_download/plantvillage dataset/color',
        '/content/kaggle_download/color',
        '/content/kaggle_download/PlantVillage',
    ]:
        if os.path.exists(candidate):
            COLOR_DIR = candidate
            break

assert COLOR_DIR, 'Folder nahi mila! Run: !find /content/kaggle_download -type d | head -20'

print(f'\n✅ Dataset mila: {COLOR_DIR}')
print(f'Total classes: {len(os.listdir(COLOR_DIR))}')

---
# 🗂️ STEP 4: Filter 11 Classes + Organize Train/Val/Test

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────
IMG_SIZE    = 224
BATCH_SIZE  = 32
SEED        = 42
DATA_DIR    = '/content/data'
MODEL_DIR   = '/content/model'

# Phase 1: train classification head (base frozen)
PHASE1_EPOCHS = 5
PHASE1_LR     = 1e-3

# Phase 2: fine-tune top layers
PHASE2_EPOCHS       = 15
PHASE2_LR           = 1e-4
UNFREEZE_FROM_LAYER = 100

os.makedirs(MODEL_DIR, exist_ok=True)
random.seed(SEED)

# ── Target Classes ────────────────────────────────────────────────────────
TARGET_CLASSES = {
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot': 'corn_gray_leaf_spot',
    'Corn_(maize)___Common_rust_':                        'corn_common_rust',
    'Corn_(maize)___healthy':                             'corn_healthy',
    'Potato___Early_blight':                              'potato_early_blight',
    'Potato___Late_blight':                               'potato_late_blight',
    'Potato___healthy':                                   'potato_healthy',
    'Tomato___Bacterial_spot':                            'tomato_bacterial_spot',
    'Tomato___Early_blight':                              'tomato_early_blight',
    'Tomato___Late_blight':                               'tomato_late_blight',
    'Tomato___Leaf_Mold':                                 'tomato_leaf_mold',
    'Tomato___healthy':                                   'tomato_healthy',
}

# Alphabetically sorted — this determines model output index (0 to 10)
LABELS_MAP  = sorted(TARGET_CLASSES.values())
NUM_CLASSES = len(LABELS_MAP)

print(f'Classes ({NUM_CLASSES}):')
for i, k in enumerate(LABELS_MAP):
    print(f'  [{i}] {k}')

# Verify folders exist
print('\nVerifying folders:')
for folder, key in TARGET_CLASSES.items():
    exists = os.path.exists(os.path.join(COLOR_DIR, folder))
    print(f'  {"✅" if exists else "❌"} {folder}')

In [ ]:
# ── Organize into train/val/test directories (80/10/10) ───────────────────
print('Organizing files into train/val/test...')

split_stats = {}
total_train = total_val = total_test = 0

for folder_name, class_key in TARGET_CLASSES.items():
    src = os.path.join(COLOR_DIR, folder_name)
    images = [f for f in os.listdir(src)
               if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    random.shuffle(images)
    n = len(images)
    n_val  = max(1, int(n * 0.10))
    n_test = max(1, int(n * 0.10))
    n_train = n - n_val - n_test

    splits = {
        'train': images[:n_train],
        'val':   images[n_train:n_train + n_val],
        'test':  images[n_train + n_val:],
    }
    for split_name, split_imgs in splits.items():
        dest = os.path.join(DATA_DIR, split_name, class_key)
        os.makedirs(dest, exist_ok=True)
        for img in split_imgs:
            shutil.copy(os.path.join(src, img), os.path.join(dest, img))

    split_stats[class_key] = {'train': n_train, 'val': n_val, 'test': n_test}
    total_train += n_train; total_val += n_val; total_test += n_test
    print(f'  {class_key:<30} total={n}  train={n_train}  val={n_val}  test={n_test}')

print(f'\nTotals → train:{total_train}  val:{total_val}  test:{total_test}')

# Save labels.json
with open(f'{MODEL_DIR}/labels.json', 'w') as f:
    json.dump(LABELS_MAP, f, indent=2)
print('\n✅ labels.json saved!')

# Compute class weights for imbalanced training
class_counts  = [split_stats[k]['train'] for k in LABELS_MAP]
total_tr      = sum(class_counts)
class_weights = {i: total_tr / (NUM_CLASSES * c) for i, c in enumerate(class_counts)}

# Save split info
split_info = {
    'labels': LABELS_MAP, 'num_classes': NUM_CLASSES,
    'train': total_train, 'val': total_val, 'test': total_test,
    'class_weights': {str(k): float(v) for k, v in class_weights.items()},
    'class_counts': class_counts,
}
with open(f'{MODEL_DIR}/split_info.json', 'w') as f:
    json.dump(split_info, f, indent=2)
print('✅ Data organized!')

In [ ]:
# Sample images visualization
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for i, class_key in enumerate(LABELS_MAP[:12]):
    ax = axes[i // 4][i % 4]
    d  = os.path.join(DATA_DIR, 'train', class_key)
    ax.imshow(plt.imread(os.path.join(d, os.listdir(d)[0])))
    ax.set_title(class_key.replace('_', '\n'), fontsize=7)
    ax.axis('off')
plt.suptitle('Sample Training Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/sample_images.png', dpi=100)
plt.show()
print('✅ Sample images shown!')

---
# 🧠 STEP 5: Build Data Pipeline

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

# Verify class folder names match LABELS_MAP (must be alphabetical)
train_classes = sorted(os.listdir(f'{DATA_DIR}/train'))
assert train_classes == LABELS_MAP, f'Order mismatch!\nFolders: {train_classes}\nExpected: {LABELS_MAP}'
print('✅ Class order verified — labels are correct')

def make_dataset(split, augment=False):
    ds = tf.keras.utils.image_dataset_from_directory(
        f'{DATA_DIR}/{split}',
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=None,
        label_mode='categorical',
        shuffle=(split == 'train'),
        seed=SEED,
        class_names=LABELS_MAP,
    )
    def preprocess(img, label):
        if augment:
            img = tf.image.random_flip_left_right(img)
            img = tf.image.random_flip_up_down(img)
            img = tf.image.random_brightness(img, 0.2)
            img = tf.image.random_contrast(img, 0.8, 1.2)
            img = tf.image.random_saturation(img, 0.8, 1.2)
            img = tf.image.random_crop(img, [int(IMG_SIZE*0.85), int(IMG_SIZE*0.85), 3])
            img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
        img = tf.keras.applications.mobilenet_v2.preprocess_input(img)
        return img, label
    return (
        ds
        .map(preprocess, num_parallel_calls=AUTOTUNE)
        .batch(BATCH_SIZE)
        .prefetch(AUTOTUNE)
    )

ds_train = make_dataset('train', augment=True)
ds_val   = make_dataset('val',   augment=False)
ds_test  = make_dataset('test',  augment=False)

imgs, lbls = next(iter(ds_train))
print(f'Batch: images={imgs.shape}  labels={lbls.shape}')
print(f'Pixel range: [{imgs.numpy().min():.2f}, {imgs.numpy().max():.2f}] (should be ~-1 to 1)')
print('✅ Data pipeline ready!')

---
# 🤖 STEP 6: Build Model (MobileNetV2)

In [ ]:
def build_model(num_classes, trainable_base=False):
    base = tf.keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet',
    )
    base.trainable = trainable_base

    inputs  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x       = base(inputs, training=trainable_base)
    x       = tf.keras.layers.GlobalAveragePooling2D()(x)
    x       = tf.keras.layers.BatchNormalization()(x)
    x       = tf.keras.layers.Dense(256, activation='relu')(x)
    x       = tf.keras.layers.Dropout(0.35)(x)
    x       = tf.cast(x, tf.float32)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax', dtype='float32')(x)

    return tf.keras.Model(inputs, outputs), base

model, base_model = build_model(NUM_CLASSES, trainable_base=False)

trainable = sum(tf.size(w).numpy() for w in model.trainable_weights)
total     = sum(tf.size(w).numpy() for w in model.weights)
print(f'Trainable params: {trainable:,} / Total: {total:,}')
print('✅ Model built!')

---
# 🏋️ STEP 7: Phase 1 Training — Head Only (~10 min)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(PHASE1_LR),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy'],
)

cb1 = [
    tf.keras.callbacks.ModelCheckpoint(
        f'{MODEL_DIR}/best_phase1.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=2, verbose=1),
]

print(f'Phase 1: {PHASE1_EPOCHS} epochs, base frozen...')
hist1 = model.fit(
    ds_train, epochs=PHASE1_EPOCHS,
    validation_data=ds_val,
    class_weight=class_weights,
    callbacks=cb1, verbose=1,
)
print(f'\n✅ Phase 1 done! Best val accuracy: {max(hist1.history["val_accuracy"])*100:.2f}%')

---
# 🔬 STEP 8: Phase 2 Fine-Tuning (~35 min)

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:UNFREEZE_FROM_LAYER]:
    layer.trainable = False

n_unfrozen = sum(1 for l in base_model.layers if l.trainable)
print(f'Unfrozen {n_unfrozen} base layers (from index {UNFREEZE_FROM_LAYER}+)')

model.compile(
    optimizer=tf.keras.optimizers.Adam(PHASE2_LR),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy'],
)

cb2 = [
    tf.keras.callbacks.ModelCheckpoint(
        f'{MODEL_DIR}/crop_disease_model.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=4,
        restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=2, verbose=1),
]

print(f'Phase 2: up to {PHASE2_EPOCHS} more epochs, fine-tuning...')
hist2 = model.fit(
    ds_train,
    initial_epoch=PHASE1_EPOCHS,
    epochs=PHASE1_EPOCHS + PHASE2_EPOCHS,
    validation_data=ds_val,
    class_weight=class_weights,
    callbacks=cb2, verbose=1,
)

model.save(f'{MODEL_DIR}/crop_disease_model.h5')
print(f'\n✅ Phase 2 done! Best val accuracy: {max(hist2.history["val_accuracy"])*100:.2f}%')
print(f'Model saved: {MODEL_DIR}/crop_disease_model.h5')

---
# 📈 STEP 9: Training Curves

In [ ]:
all_acc      = hist1.history['accuracy']     + hist2.history['accuracy']
all_val_acc  = hist1.history['val_accuracy'] + hist2.history['val_accuracy']
all_loss     = hist1.history['loss']         + hist2.history['loss']
all_val_loss = hist1.history['val_loss']     + hist2.history['val_loss']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for ax, train, val, title in [
    (ax1, all_acc, all_val_acc, 'Accuracy'),
    (ax2, all_loss, all_val_loss, 'Loss'),
]:
    ax.plot(train, label='Train', color='#16a34a', lw=2)
    ax.plot(val,   label='Val',   color='#f59e0b', lw=2)
    ax.axvline(x=PHASE1_EPOCHS-0.5, color='gray', ls='--', label='Fine-tune start')
    ax.set_title(title, fontsize=13); ax.set_xlabel('Epoch')
    ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('Training Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/training_curves.png', dpi=120)
plt.show()
print('✅ Training curves saved!')

---
# ⚡ STEP 10: Convert to TFLite (INT8) + TensorFlow.js

In [ ]:
# Build calibration dataset from val images
print('Building calibration dataset...')
calib_ds = tf.keras.utils.image_dataset_from_directory(
    f'{DATA_DIR}/val',
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=None, shuffle=True, seed=42,
    class_names=LABELS_MAP,
)

calib_images = list(
    calib_ds.take(200).map(
        lambda img, lbl: tf.expand_dims(
            tf.keras.applications.mobilenet_v2.preprocess_input(
                tf.cast(img, tf.float32)
            ), 0
        )
    )
)

def representative_dataset():
    for sample in calib_images:
        yield [sample]

# TFLite INT8 conversion
TFLITE_PATH = f'{MODEL_DIR}/crop_disease_model.tflite'
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8,
    tf.lite.OpsSet.TFLITE_BUILTINS,
]
converter.inference_input_type  = tf.int8
converter.inference_output_type = tf.int8

print('Converting to INT8 TFLite (~3-5 min)...')
tflite_model = converter.convert()
with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_model)

orig_mb   = os.path.getsize(f'{MODEL_DIR}/crop_disease_model.h5') / 1e6
tflite_mb = os.path.getsize(TFLITE_PATH) / 1e6
print(f'\n✅ TFLite done! {orig_mb:.1f} MB → {tflite_mb:.1f} MB ({orig_mb/tflite_mb:.1f}x smaller)')

In [ ]:
# Convert to TensorFlow.js (float16 for browser inference)
os.makedirs('/content/tfjs_model', exist_ok=True)

print('Converting to TensorFlow.js...')
!tensorflowjs_converter \
    --input_format=keras \
    --output_format=tfjs_layers_model \
    --quantize_float16 \
    /content/model/crop_disease_model.h5 \
    /content/tfjs_model/

# labels.json also goes into tfjs folder (app reads it)
shutil.copy(f'{MODEL_DIR}/labels.json', '/content/tfjs_model/labels.json')

tfjs_mb = sum(
    os.path.getsize(os.path.join('/content/tfjs_model', f))
    for f in os.listdir('/content/tfjs_model')
) / 1e6
print(f'\n✅ TFJS done! Total size: {tfjs_mb:.1f} MB')
!ls -lh /content/tfjs_model/

---
# 📊 STEP 11: Evaluate — Accuracy, F1, Confusion Matrix

In [ ]:
print('Running inference on test set...')
y_true, y_pred, y_conf = [], [], []

for images, labels in ds_test:
    preds        = model.predict(images, verbose=0)
    pred_classes = np.argmax(preds, axis=1)
    true_classes = np.argmax(labels.numpy(), axis=1) \
        if len(labels.shape) > 1 else labels.numpy()
    y_true.extend(true_classes.tolist())
    y_pred.extend(pred_classes.tolist())
    y_conf.extend(np.max(preds, axis=1).tolist())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

top1_acc = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average='macro')

print(f'\n{"="*50}')
print(f'  ⭐ FINAL RESULTS — Resume mein ye numbers daalo!')
print(f'{"="*50}')
print(f'  Top-1 Accuracy : {top1_acc*100:.2f}%')
print(f'  Macro F1-Score : {macro_f1:.4f}')
print(f'  Test Samples   : {len(y_true)}')
print(f'{"="*50}')

report = classification_report(y_true, y_pred, target_names=LABELS_MAP, digits=4)
print('\nPer-class report:')
print(report)

with open(f'{MODEL_DIR}/classification_report.txt', 'w') as f:
    f.write(f'Top-1 Accuracy: {top1_acc*100:.2f}%\nMacro F1: {macro_f1:.4f}\n\n')
    f.write(report)

In [ ]:
# Confusion Matrix
cm      = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
short   = ['C-GLS','C-Rust','C-H','P-EB','P-LB','P-H',
           'T-BS','T-EB','T-LB','T-LM','T-H']

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=short, yticklabels=short,
            linewidths=0.5, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix (Acc: {top1_acc*100:.2f}%)', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/confusion_matrix.png', dpi=150)
plt.show()

# F1 Bar Chart
per_class_f1 = f1_score(y_true, y_pred, average=None)
colors = ['#16a34a' if f>=0.95 else '#f59e0b' if f>=0.85 else '#ef4444'
          for f in per_class_f1]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(LABELS_MAP, per_class_f1, color=colors)
ax.set_xlim(0, 1.05)
ax.axvline(x=macro_f1, color='gray', ls='--', label=f'Macro avg: {macro_f1:.3f}')
ax.set_title('Per-Class F1-Score', fontweight='bold'); ax.legend()
for bar, f in zip(bars, per_class_f1):
    ax.text(bar.get_width()+0.01, bar.get_y()+bar.get_height()/2,
            f'{f:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/f1_per_class.png', dpi=120)
plt.show()

metrics = {
    'test_accuracy': round(float(top1_acc), 4),
    'macro_f1': round(float(macro_f1), 4),
    'test_samples': int(len(y_true)),
    'per_class_f1': {k: round(float(v), 4) for k, v in zip(LABELS_MAP, per_class_f1)},
}
with open(f'{MODEL_DIR}/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('✅ Evaluation complete!')

---
# 📥 STEP 12: Download Everything

In [ ]:
print('Packaging files...')

shutil.make_archive('/content/tfjs_model_for_app', 'zip', '/content/tfjs_model')
shutil.make_archive('/content/model_artifacts',    'zip', MODEL_DIR)

print('\n📦 2 files download honge:')
print('  1. tfjs_model_for_app.zip → unzip karke app/public/model/ mein daalo')
print('  2. model_artifacts.zip    → TFLite + accuracy charts')
print()

from google.colab import files
files.download('/content/tfjs_model_for_app.zip')
files.download('/content/model_artifacts.zip')

print(f'\n🎉 PIPELINE COMPLETE!')
print(f'   Accuracy: {top1_acc*100:.2f}%')
print(f'   F1 Score: {macro_f1:.4f}')
print(f'\nNext: unzip tfjs_model_for_app.zip → copy to app/public/model/ → npm run dev')